In [1]:
import torch
from torch.nn.functional import sigmoid
from PIL import Image
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms.functional import to_pil_image

In [2]:
rn18 = resnet18(weights='DEFAULT')
rn18.eval()
tfm = ResNet18_Weights.IMAGENET1K_V1.transforms()


featmaps = {}

def build_save_featmap(name):
    def hook(module, args, output):
        if name in ['avgpool', 'fc']:
            return
        output = output[0]
        dim = list(range(1, output.ndim))
        mu = output.mean(dim=dim, keepdim=True)
        sigma = output.std(dim=dim, keepdim=True)
        output = sigmoid((output - mu)/(sigma + 1e-5))
        featmaps[name] = output
    return hook

for name, layer in rn18.named_children():
    layer.register_forward_hook(build_save_featmap(name))

In [3]:
im = Image.open('images/waldek1.jpg')
t_input = tfm(im)

with torch.no_grad():
    out = rn18(t_input[None, ...])

In [4]:
def tensor_to_js(arr, name):
    lines = []
    lines.append(f'const {name} = [')

    for channel in arr:
        lines.append(' '*2 + '[')
        for row in channel:
            lines.append(' '*4 + '[' + r', '.join(fr'{x}' for x in row) + '],')
        lines.append(' '*2 + '],')

    lines.append('];')

    lines.append('')
    lines.append(f'export default {name};')
    return '\n'.join(lines)

In [5]:
from pathlib import Path

num_maps = {
    'conv1': 2,
    'layer1': 4,
    'layer2': 8,
    'layer3': 16,
    'layer4': 32,
}

for name, num in num_maps.items():
    t = featmaps[name][:num]
    text = tensor_to_js(t, name)
    Path(f'{name}.js').write_text(text)

    for i, ch in enumerate(t):
        im = to_pil_image(ch)
        im.save(f'activation_images/{name}_{i:02}.png')

In [6]:
to_pil_image(t_input).save('activation_images/input.png')